In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [ ]:
load_dotenv(override=True)

In [ ]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

### Strategy for the Deep Research Agent
We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to Runner.run() for each step in the process.

We will use Structured Outputs at each point.

### We will build 4 Agents:
The Search Agent: searches the web for information
The Planner Agent: given a question, comes up with a list of searches that should be made
The Writer Agent: writes a robust report
The Emailer Agent: crafts and sends an email
And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.

Agent 1: The Search Agent
OpenAI Hosted Tools
https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

WebSearchTool lets an agent search the web.
FileSearchTool allows retrieving information from your OpenAI Vector Stores.
CodeInterpreterTool lets the LLM execute code in a sandboxed environment.
HostedMCPTool exposes a remote MCP server's tools to the model.
ImageGenerationTool generates images from a prompt.
ToolSearchTool lets the model load deferred tools, namespaces, or hosted MCP servers on demand.

Important note - API charge of WebSearchTool
This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing



In [ ]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [ ]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)    

In [ ]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))